# ETL — Modelagem Dimensional Olist

Este notebook implementa o pipeline ETL que transforma os CSVs brutos do Olist
em tabelas alinhadas ao modelo dimensional Star Schema definido no projeto.

**Saída:** 6 arquivos Parquet em `data/processed/`

| Tabela | Tipo | Descrição |
|--------|------|-----------|
| `d_category` | Dimensão | Categorias de produto |
| `d_payment` | Dimensão | Métodos de pagamento |
| `d_state` | Dimensão conformada | Estados de clientes e vendedores |
| `f_orders` | Fato | Pedidos com métricas de entrega e avaliação |
| `f_order_items` | Fato | Itens vendidos com preço e categoria |
| `f_payments` | Fato | Transações de pagamento |

> **Pré-requisito:** Baixar o dataset em https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
> e extrair os CSVs para `data/raw/`.

In [1]:
import pandas as pd
import numpy as np

## 1. Carregamento dos dados brutos

Leitura dos 7 CSVs originais sem nenhuma modificação.
Os dados brutos nunca são alterados — todas as transformações criam novos DataFrames.

In [2]:
df_orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_reviews   = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_payments  = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
df_products  = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_sellers   = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

print("CSVs carregados com sucesso.")

CSVs carregados com sucesso.


## 2. Correção de tipos de dados

As colunas de data são lidas como string pelo Pandas.
Convertendo para datetime para permitir o cálculo de `time_delivered`.

In [3]:
df_orders['order_purchase_timestamp']      = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])

## 3. Tratamento de nulos — df_products

Durante a exploração foram identificados registros fantasma em `df_products`:
linhas onde apenas `product_id` está preenchido e todas as demais colunas são nulas.
Esses registros não têm valor analítico e são removidos antes de criar `d_category`.

In [4]:
colunas_sem_id = [col for col in df_products.columns if col != 'product_id']
df_products    = df_products.dropna(subset=colunas_sem_id, how='all')

print(f"df_products após remoção de registros fantasma: {df_products.shape[0]} linhas")

df_products após remoção de registros fantasma: 32950 linhas


## 4. Criação das tabelas dimensão

As dimensões são criadas a partir dos valores únicos de cada atributo descritivo.
Cada dimensão recebe um ID surrogate sequencial gerado pelo pipeline.

**`d_state`** é uma dimensão conformada — agrupa estados de clientes e vendedores
em uma única tabela, permitindo que `f_orders` e `f_order_items` compartilhem
o mesmo contexto geográfico sem duplicar a dimensão.

In [5]:
d_category = df_products[['product_category_name']].dropna().drop_duplicates().reset_index(drop=True)
d_category['category_id'] = d_category.index + 1
d_category = d_category[['category_id', 'product_category_name']]

print(f"d_category: {len(d_category)} categorias")

d_category: 73 categorias


In [6]:
d_payment = df_payments[['payment_type']].dropna().drop_duplicates().reset_index(drop=True)
d_payment['payment_id'] = d_payment.index + 1
d_payment = d_payment[['payment_id', 'payment_type']]

print(f"d_payment: {len(d_payment)} métodos de pagamento")

d_payment: 5 métodos de pagamento


In [7]:
# Dimensão conformada: une estados de clientes e vendedores sem duplicatas
estados   = pd.concat([df_customers['customer_state'], df_sellers['seller_state']]).dropna().drop_duplicates().reset_index(drop=True)
d_state   = pd.DataFrame({'customer_seller_state': estados})
d_state['state_id'] = d_state.index + 1
d_state   = d_state[['state_id', 'customer_seller_state']]

print(f"d_state: {len(d_state)} estados únicos")

d_state: 27 estados únicos


## 5. Criação de f_payments

Tabela fato com granularidade de transação de pagamento — uma linha por evento.
Um pedido pode ter múltiplos registros (ex: parte em cartão, parte em voucher).

`order_purchase_timestamp` é trazido de `df_orders` via `order_id` para permitir
análise temporal de faturamento no Power BI via `d_calendar`.

`order_id` permanece como dimensão degenerada — chave de transação para agrupamento,
sem criar relacionamento direto com `f_orders`.

In [8]:
f_payments = df_payments.merge(d_payment, on='payment_type', how='left')
f_payments = f_payments.merge(
    df_orders[['order_id', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

f_payments = f_payments.reset_index(drop=True)
f_payments['payment_transaction_id'] = f_payments.index + 1

f_payments = f_payments[[
    'payment_transaction_id',
    'payment_installments',
    'payment_value',
    'order_id',
    'payment_id',
    'order_purchase_timestamp'
]]

print(f"f_payments: {f_payments.shape[0]} transações | {f_payments['order_purchase_timestamp'].isna().sum()} nulos em order_purchase_timestamp")

f_payments: 103886 transações | 0 nulos em order_purchase_timestamp


## 6. Criação de f_orders

Tabela fato com granularidade de pedido — uma linha por pedido.

**Colunas calculadas:**
- `time_delivered`: dias entre a compra e a entrega, arredondado para cima com `np.ceil`.
  Do ponto de vista do cliente, 8.1 dias de espera equivalem a 9 dias.
  Calculado apenas para pedidos entregues — nulos em pedidos cancelados ou em trânsito
  são esperados e mantidos com tipo `Int64` para suportar valores ausentes.

- `review_score`: média das avaliações por pedido. Um pedido pode ter múltiplas avaliações
  no dataset original — a média evita duplicação de linhas.

`customer_state_id` é a chave estrangeira para `d_state` representando o estado do cliente.

In [9]:
df_reviews_agg = df_reviews.groupby('order_id', as_index=False)['review_score'].mean()

f_orders = df_orders.merge(df_customers[['customer_id', 'customer_state']], on='customer_id', how='left')
f_orders = f_orders.merge(d_state, left_on='customer_state', right_on='customer_seller_state', how='left')
f_orders = f_orders.rename(columns={'state_id': 'customer_state_id'})
f_orders = f_orders.merge(df_reviews_agg, on='order_id', how='left')

f_orders['time_delivered'] = np.ceil(
    (f_orders['order_delivered_customer_date'] - f_orders['order_purchase_timestamp']) / pd.Timedelta(days=1)
).astype('Int64')

f_orders = f_orders[[
    'order_id',
    'time_delivered',
    'order_delivered_customer_date',
    'review_score',
    'customer_state_id',
    'order_purchase_timestamp'
]]

print(f"f_orders: {f_orders.shape[0]} pedidos")

f_orders: 99441 pedidos


## 7. Criação de f_order_items

Tabela fato com granularidade de item — uma linha por produto dentro de um pedido.

`seller_state_id` é a chave estrangeira para `d_state` representando o estado do vendedor.
A mesma dimensão `d_state` é compartilhada com `f_orders` via `customer_state_id` —
padrão de dimensão conformada que elimina redundância no modelo.

`order_id` permanece como dimensão degenerada para agrupamento por pedido
sem criar relacionamento fato-fato com `f_orders`.

In [10]:
f_order_items = df_items.merge(df_products[['product_id', 'product_category_name']], on='product_id', how='left')
f_order_items = f_order_items.merge(d_category, on='product_category_name', how='left')
f_order_items = f_order_items.merge(df_sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
f_order_items = f_order_items.merge(d_state, left_on='seller_state', right_on='customer_seller_state', how='left')
f_order_items = f_order_items.rename(columns={'state_id': 'seller_state_id'})

f_order_items = f_order_items.reset_index(drop=True)
f_order_items['item_transaction_id'] = f_order_items.index + 1

f_order_items = f_order_items[[
    'item_transaction_id',
    'product_id',
    'price',
    'seller_state_id',
    'category_id',
    'order_id'
]]

print(f"f_order_items: {f_order_items.shape[0]} itens")

f_order_items: 112650 itens


## 8. Validação do pipeline

Verificações obrigatórias antes de exportar:
1. Volumetria de `f_orders` deve ser igual a `df_orders` — nenhuma linha perdida ou duplicada
2. Volumetria de `f_order_items` deve ser igual a `df_items`
3. Chaves estrangeiras críticas não podem ter nulos

In [11]:
print("Iniciando validação de dados...")

if len(f_orders) != len(df_orders):
    raise ValueError(f"ERRO: f_orders tem {len(f_orders)} linhas, mas df_orders tem {len(df_orders)}.")

if len(f_order_items) != len(df_items):
    raise ValueError(f"ERRO: f_order_items tem {len(f_order_items)} linhas, mas df_items tem {len(df_items)}.")

nulos_estado_cliente  = f_orders['customer_state_id'].isna().sum()
nulos_estado_vendedor = f_order_items['seller_state_id'].isna().sum()

if nulos_estado_cliente > 0:
    raise ValueError(f"ERRO: {nulos_estado_cliente} nulos em customer_state_id na f_orders.")

if nulos_estado_vendedor > 0:
    raise ValueError(f"ERRO: {nulos_estado_vendedor} nulos em seller_state_id na f_order_items.")

print("Validação concluída: volumetria preservada e chaves estrangeiras íntegras.")

Iniciando validação de dados...
Validação concluída: volumetria preservada e chaves estrangeiras íntegras.


## 9. Exportação para Parquet

Parquet foi escolhido por ser um formato colunar comprimido, mais eficiente
para leitura analítica do que CSV — especialmente no Power BI com grandes volumes.

Ordem de exportação: dimensões primeiro, depois tabelas fato.

In [12]:
print("Exportando arquivos Parquet...")

# Dimensões
d_category.to_parquet('../data/processed/d_category.parquet',  index=False)
d_payment.to_parquet('../data/processed/d_payment.parquet',    index=False)
d_state.to_parquet('../data/processed/d_state.parquet',        index=False)

# Fatos
f_payments.to_parquet('../data/processed/f_payments.parquet',        index=False)
f_orders.to_parquet('../data/processed/f_orders.parquet',            index=False)
f_order_items.to_parquet('../data/processed/f_order_items.parquet',  index=False)

print("ETL concluído. 6 arquivos Parquet gerados em data/processed/")

Exportando arquivos Parquet...
ETL concluído. 6 arquivos Parquet gerados em data/processed/
